# Reproducing Tables 3 & 4 of *Mixup for Node and Graph Classification* (Wang et al., WWW '21)

**Project:** COMP8240 – Reproducing and Extending Mixup for Node Classification in GNNs
**Paper:** Y. Wang, W. Wang, Y. Liang, Y. Cai, B. Hooi. *Mixup for Node and Graph Classification.* WWW 2021, pp. 3663–3674.
**Reference code:** `github.com/vanoracai/MixupForGraph` (`mixup.py`, `graph_conv.py`, MIT licence)

This notebook reproduces the two node-classification tables of the paper:

| Paper table | Setting | Datasets | Metric | Methods run here |
|---|---|---|---|---|
| **Table 3** | Transductive, random 60/20/20 splits | Citeseer, Cora, Pubmed | Test accuracy (%) | GCN, GAT, JKNet, DropEdge+GCN, DropEdge+JKNet, **Mixup+GCN**, **Mixup+JKNet** |
| **Table 4** | Inductive, fixed GraphSAINT splits | Flickr, Yelp, Amazon | Test F1-micro (%) | GraphSAGE-mean, DropEdge+GraphSAGE, **Mixup+GraphSAGE**, GraphSAINT-GCN, GraphSAINT-JKNet, DropEdge+GraphSAINT, **Mixup+GraphSAINT** |

Baselines that are not re-implemented (LGCN, GMNN, ResGCN, GraphSAGE-LSTM/pool, GraphSAINT-GAT) are shown with the paper's numbers for reference only.

### How the notebook is organised
1. **Environment setup** – GPU check, install, imports
2. **Configuration** – one place to choose the run mode, datasets, methods and hyper-parameters
3. **Core utilities** – sparse aggregation, DropEdge, metrics, result cache
4. **Models** – two-branch / two-stage Mixup GNN (Algorithm 1 of the paper), GAT baseline
5. **Sanity check** – our implementation vs. the authors' released code (must match numerically)
6. **Table 3** – transductive experiments + comparison with the paper (includes a **one-minute demo** for the update video)
7. **Authors' repository configuration** – continuity with the earlier Pubmed pilot run
8. **New data (existing datasets)** – the same Table 3 protocol on Amazon Computers and Amazon Photo
9. **Table 4** – inductive experiments + comparison with the paper
10. **Final comparison, figures and export**

### Before you run
* **Runtime → Change runtime type → GPU** (T4 is enough for Table 3, the new datasets, Flickr and Yelp).
* Default settings: **medium** protocol for Table 3, **quick** protocol for Table 4 (see Section 2). Rough T4 time: Table 3 ≈ 2–3 h, Amazon Computers/Photo ≈ 1 h, Table 4 (Flickr + Yelp) ≈ 1.5–2 h. These are estimates; the first runs print their time so you can check.
* Amazon **Products** (Table 4, 1.6 M nodes, 264 M directed edges) needs **High-RAM** (Colab Pro, ~40 GB RAM) and is switched **off** by default. (Not to be confused with Amazon Computers/Photo in Section 8, which are small.)
* Every finished run is appended to a results file. If Colab disconnects, re-run the cells and finished runs are skipped automatically. **Set `USE_DRIVE = True`** in Section 2 to keep results across sessions.
* You can run sections independently: the `RUN_TABLE3`, `RUN_REPO_CONFIG`, `RUN_NEW_DATA` and `RUN_TABLE4` switches in Section 2 turn each part on or off.

---
## 1. Environment setup
### 1.1 Check the GPU

In [ ]:
!nvidia-smi

### 1.2 Install PyTorch Geometric
Colab already ships a recent PyTorch. Only `torch_geometric` is needed: the notebook does all sparse aggregation and graph sampling in plain PyTorch, so it does **not** depend on `torch_scatter` / `torch_sparse` (the compiled packages that caused the version conflicts with the authors' PyTorch 1.8 / PyG 1.7 requirements).

In [ ]:
%pip install -q torch_geometric

### 1.3 Imports and version log

In [ ]:
import os, os.path as osp, sys, json, math, time, random, platform, warnings
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch_geometric
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ENV = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'torch_geometric': torch_geometric.__version__,
    'numpy': np.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
for k, v in ENV.items():
    print(f'{k:>16}: {v}')
if DEVICE.type == 'cpu':
    print('\nWARNING: no GPU detected - Table 3 will be slow and Table 4 very slow.')

---
## 2. Experiment configuration

**Protocol in the paper**
* **Table 3:** 10 random 60/20/20 splits, 100 trials with random weight initialisation, number of layers searched in 1…30 and chosen on the validation set.
* **Table 4:** fixed GraphSAINT splits, 100 trials, number of layers searched in 1…30 and chosen on the validation set.
* Mixup weight λ ~ Beta(α, α) with **α = 1** (paper default). *Note:* the released `mixup.py` uses Beta(4, 4); Section 7 runs that exact configuration.

The full protocol is thousands of training runs, so three presets are provided:

| `RUN_MODE` | Table 3 runs per (method, dataset, depth) | Table 3 depths | Table 4 seeds | Table 4 depths | Rough time on a T4 |
|---|---|---|---|---|---|
| `'quick'` | 10 splits × 1 seed = 10 | 2, 3 | 3 | 2 | ~20–30 min (T3) + ~1.5–2 h (Flickr+Yelp) |
| `'medium'` | 10 splits × 3 seeds = 30 | 2, 3, 4, 8 | 5 | 2, 3 | ~2–3 h (T3) + ~4–6 h (Flickr+Yelp) |
| `'paper'` | 10 splits × 10 seeds = 100 | 1, 2, 3, 4, 8, 16, 30 | 100 | 1, 2, 3, 4 | 12 h+ (T3), a day+ (T4); several sessions |

**Default used for the project update:** `RUN_MODE = 'medium'` for Table 3 (the core of the reproduction), with `T4_OVERRIDE` switching Table 4 to the quick settings. Set `T4_OVERRIDE = {}` to use the same preset for both tables.

Hyper-parameters for Table 3 follow the authors' released code (hidden 256, dropout 0.4, Adam lr 1e-3, 300 epochs). Table 4 follows the GraphSAINT settings for each dataset. Model selection everywhere uses **validation** accuracy / F1, and the test score at the best validation epoch is reported.

In [ ]:
RUN_MODE = 'medium'           # 'quick' | 'medium' | 'paper'   (applies to Table 3)
T4_OVERRIDE = dict(t4_seeds=3, t4_depths=[2])   # quick settings for Table 4; set to {} to follow RUN_MODE

PRESETS = {
    'quick':  dict(t3_splits=10, t3_seeds=1,  t3_depths=[2, 3],                    t4_seeds=3,   t4_depths=[2]),
    'medium': dict(t3_splits=10, t3_seeds=3,  t3_depths=[2, 3, 4, 8],              t4_seeds=5,   t4_depths=[2, 3]),
    'paper':  dict(t3_splits=10, t3_seeds=10, t3_depths=[1, 2, 3, 4, 8, 16, 30],  t4_seeds=100, t4_depths=[1, 2, 3, 4]),
}
P = dict(PRESETS[RUN_MODE], **T4_OVERRIDE)

MIXUP_ALPHA = 1.0             # λ ~ Beta(α, α); paper default α = 1
DROPEDGE_P  = 0.3             # fraction of edges removed per training step for DropEdge baselines

# ---------------- Table 3 (transductive) ----------------
RUN_TABLE3 = True
T3_DATASETS = ['Citeseer', 'Cora', 'Pubmed']
T3_METHODS  = ['GCN', 'GAT', 'JKNet', 'DropEdge+GCN', 'DropEdge+JKNet', 'Mixup+GCN', 'Mixup+JKNet']
T3_CONV     = 'gcn'           # 'gcn' = Kipf & Welling GCN layer (paper); 'graphconv' = layer used in the authors' repo
T3_HP  = dict(hidden=256, dropout=0.4, lr=1e-3, weight_decay=0.0, epochs=300)       # authors' mixup.py settings
GAT_HP = dict(hidden=8, heads=8, dropout=0.6, lr=5e-3, weight_decay=5e-4, epochs=300)  # Velickovic et al. settings

# ---------------- Section 7: authors' repo configuration ----------------
RUN_REPO_CONFIG = True
REPO_DATASETS   = ['Pubmed', 'Cora', 'Citeseer']
REPO_SEEDS      = 5           # the repo uses one fixed seed (0); several seeds give a mean ± std

# ---------------- Section 8: new data (existing datasets not used in the paper) ----------------
RUN_NEW_DATA = True
NEW_DATASETS = ['Computers', 'Photo']      # Amazon co-purchase graphs (Shchur et al., 2018); add 'CS' / 'Physics' (Coauthor) if time allows
NEW_METHODS  = ['GCN', 'Mixup+GCN', 'JKNet', 'Mixup+JKNet']
NEW_SPLITS, NEW_SEEDS, NEW_DEPTHS = 10, 2, [2, 3]   # same 60/20/20 protocol as Table 3, 20 runs per setting
NEW_EPOCHS = 500              # Amazon Photo was still improving at epoch 300 in a pilot run; no paper setting exists here

# ---------------- Table 4 (inductive) ----------------
RUN_TABLE4  = True
RUN_AMAZON  = False           # needs Colab High-RAM (~40 GB); set True only on a High-RAM runtime
T4_DATASETS = ['Flickr', 'Yelp'] + (['Amazon'] if RUN_AMAZON else [])
T4_METHODS  = ['GraphSAGE-mean', 'DropEdge+GraphSAGE', 'Mixup+GraphSAGE',
               'GraphSAINT-GCN', 'GraphSAINT-JKNet', 'DropEdge+GraphSAINT', 'Mixup+GraphSAINT']
T4_HP = {   # per-dataset settings (GraphSAINT paper / PyG examples, adapted to a single Colab GPU)
    'Flickr': dict(hidden=256, dropout=0.2, lr=1e-2, epochs=60, eval_every=2,
                   saint_roots=6000, saint_walk=2, sage_batch=1024, sage_fanouts=[25, 10, 10, 5]),
    'Yelp':   dict(hidden=512, dropout=0.1, lr=1e-2, epochs=30, eval_every=2,
                   saint_roots=3000, saint_walk=2, sage_batch=2048, sage_fanouts=[25, 10, 10, 5]),
    'Amazon': dict(hidden=256, dropout=0.1, lr=1e-2, epochs=20, eval_every=2,
                   saint_roots=3000, saint_walk=2, sage_batch=2048, sage_fanouts=[25, 10, 10, 5]),
}
USE_SAINT_NORM      = True    # GraphSAINT aggregator/loss normalisation (Zeng et al., 2020)
SAINT_COVERAGE      = 50      # avg. times each training node is sampled when estimating the normalisation
MAX_STEPS_PER_EPOCH = None    # e.g. 100 to cap mini-batches per epoch on very large graphs

# ---------------- storage ----------------
USE_DRIVE = False             # True: keep results (and optionally data) on Google Drive
print('Run mode:', RUN_MODE, P)
print('Table 4 datasets:', T4_DATASETS)

### 2.1 Storage and result cache
Each training run is written as one JSON line to `results.jsonl`. Finished runs are skipped if the cell is re-run, so a disconnect only costs the run in progress.

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/mixup_reproduction'
else:
    OUT_DIR = './mixup_reproduction'
DATA_DIR = './data'            # datasets are re-downloaded quickly; keep them on the local disk
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)


class ResultStore:
    """Append-only JSON-lines store keyed by a run identifier."""
    def __init__(self, path):
        self.path, self.rows = path, {}
        if osp.exists(path):
            with open(path) as f:
                for line in f:
                    if line.strip():
                        r = json.loads(line)
                        self.rows[r['key']] = r

    def __contains__(self, key):
        return key in self.rows

    def put(self, key, rec):
        rec = dict(rec, key=key)
        self.rows[key] = rec
        with open(self.path, 'a') as f:
            f.write(json.dumps(rec) + '\n')

    def frame(self, prefix):
        rows = [r for k, r in self.rows.items() if k.startswith(prefix)]
        return pd.DataFrame(rows)


STORE = ResultStore(osp.join(OUT_DIR, 'results.jsonl'))
print(f'Results file: {STORE.path}  ({len(STORE.rows)} runs already stored)')

---
## 3. Core utilities

**Sparse aggregation.** A GNN layer computes, for every node *i*, an aggregate of its neighbours plus a self term. We store the neighbour part as a sparse matrix **A** (row *i* = the neighbours whose messages node *i* receives) and the self part as a per-node weight:
* `gcn` (Kipf & Welling): $\hat{A}=D^{-1/2}(A+I)D^{-1/2}$ split into neighbour weights $1/\sqrt{(d_i+1)(d_j+1)}$ and self weight $1/(d_i+1)$.
* `graphconv` (authors' repo / GraphSAGE-mean): mean over neighbours with weight $1/d_i$ and a separate root linear map.

Keeping the self term separate is exactly what the two-branch Mixup convolution needs: the neighbour messages come from the stage-one representations, while the self term uses the mixed representation (Eq. 6). Very large graphs (Amazon) are multiplied in row blocks so the adjacency never has to sit on the GPU in one piece.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def edge_index_to_csr(edge_index, num_nodes):
    """CSR with rows = target nodes (edge_index[1]) and columns = source nodes (edge_index[0]).
    Self-loops and duplicate edges are removed."""
    src, tgt = edge_index[0].long(), edge_index[1].long()
    key = tgt * num_nodes + src
    key = torch.unique(key[src != tgt])          # sorted by (target, source), no duplicates
    tgt, src = key // num_nodes, key % num_nodes
    rowptr = torch.zeros(num_nodes + 1, dtype=torch.long)
    rowptr[1:] = torch.cumsum(torch.bincount(tgt, minlength=num_nodes), 0)
    return rowptr, src


def rowptr_from_rows(rows, n):
    rowptr = torch.zeros(n + 1, dtype=torch.long)
    rowptr[1:] = torch.cumsum(torch.bincount(rows, minlength=n), 0)
    return rowptr


def csr_rows(rowptr):
    return torch.repeat_interleave(torch.arange(rowptr.numel() - 1), rowptr.diff())


def agg_weights(kind, row, col, n, deg=None):
    """Neighbour edge weights and per-node self weights for one aggregation scheme."""
    if deg is None:
        deg = torch.bincount(row, minlength=n)
    deg = deg.float()
    if kind == 'gcn':
        dinv = (deg + 1.0).rsqrt()
        return dinv[row] * dinv[col], 1.0 / (deg + 1.0)
    if kind == 'graphconv':
        return 1.0 / deg[row].clamp(min=1.0), torch.zeros(n)
    raise ValueError(kind)


class GraphOps:
    """Sparse neighbour-aggregation operator  h -> A @ h  plus per-node self weights."""
    def __init__(self, rowptr, col, val, n, self_w, device, max_nnz=50_000_000):
        self.n, self.device = n, device
        self.self_w = self_w.to(device)
        if col.numel() <= max_nnz:          # small enough: one COO matrix on the device (supports autograd)
            idx = torch.stack([csr_rows(rowptr), col])
            self.A = torch.sparse_coo_tensor(idx, val, (n, n)).coalesce().to(device)
            self.blocks = None
        else:                               # huge graph: keep CSR on the CPU and multiply in row blocks (inference only)
            self.A = None
            self.rowptr, self.col, self.val = rowptr, col, val
            self.blocks, s = [], 0
            while s < n:
                e = int(torch.searchsorted(rowptr, rowptr[s] + max_nnz, right=True)) - 1
                e = min(max(e, s + 1), n)
                self.blocks.append((s, e))
                s = e

    def agg(self, h):
        if self.A is not None:
            return torch.sparse.mm(self.A, h)
        out = torch.empty(self.n, h.size(1), device=h.device, dtype=h.dtype)
        for s, e in self.blocks:
            p0, p1 = int(self.rowptr[s]), int(self.rowptr[e])
            crow = (self.rowptr[s:e + 1] - p0).to(h.device)
            blk = torch.sparse_csr_tensor(crow, self.col[p0:p1].to(h.device), self.val[p0:p1].to(h.device),
                                          (e - s, self.n))
            out[s:e] = blk @ h
        return out


def make_graph(rowptr, col, n, kind, device, deg=None, edge_scale=None):
    val, self_w = agg_weights(kind, csr_rows(rowptr), col, n, deg)
    if edge_scale is not None:
        val = val * edge_scale
    return GraphOps(rowptr, col, val.float(), n, self_w.float(), device)


def drop_edge_undirected(edge_index, p):
    """DropEdge (Rong et al., 2020): remove a random fraction p of undirected edges."""
    src, dst = edge_index
    und = edge_index[:, src < dst]
    und = und[:, torch.rand(und.size(1)) >= p]
    return torch.cat([und, und.flip(0)], dim=1)


@torch.no_grad()
def micro_f1(logits, y, multilabel):
    """F1-micro. For single-label tasks this equals accuracy."""
    if not multilabel:
        return (logits.argmax(1) == y).float().mean().item()
    pred, y = logits > 0, y.bool()
    tp = (pred & y).sum().item()
    fp = (pred & ~y).sum().item()
    fn = (~pred & y).sum().item()
    return 2 * tp / max(2 * tp + fp + fn, 1)

print('utilities ready')

---
## 4. Models

### 4.1 Two-branch, two-stage Mixup GNN (Algorithm 1)
For a pair of nodes *(i, j)* with *j = π(i)* (π is a random permutation of the training nodes) and λ ~ Beta(α, α):

* **Stage 1** (lines 1–8): ordinary forward pass, no Mixup → hidden representations $h^{(l)}$ for $l = 0 … L-1$.
* **Stage 2** (lines 9–20): mix the input attributes $\tilde{x}_{ij}=\lambda x_i+(1-\lambda)x_j$ (Eq. 5). At every layer run two branches that aggregate the **stage-one** neighbour representations of *i* and of *j* respectively, with the mixed representation as the self term (Eq. 6), then mix the two outputs (Eq. 7).
* **Loss** (line 21): $\lambda\,\mathcal{L}(\hat y, y_i) + (1-\lambda)\,\mathcal{L}(\hat y, y_j)$, which equals the loss against the mixed label $\tilde y_{ij}$.

Implementation note: the branch of node *j* aggregates node *j*'s neighbourhood, i.e. row π(i) of $A h^{(l)}$. Because stage-one representations are shared, the neighbour aggregation is computed once per layer and simply re-indexed by π for the second branch. This is equivalent to the authors' re-labelled `edge_index_b` (verified in Section 5).

The backbone is `num_layers` graph-convolution layers + a linear classifier (as in the authors' code). `jk='cat'` gives JKNet (Xu et al., 2018): the classifier reads the concatenation of all layer outputs. With `lam=1` and the identity permutation the Mixup forward reduces exactly to the ordinary forward, so evaluation uses the plain forward.

In [ ]:
class MixConv(nn.Module):
    """One graph-convolution layer split into a neighbour part and a self (centre) part."""
    def __init__(self, d_in, d_out, kind):
        super().__init__()
        self.kind = kind
        self.lin = nn.Linear(d_in, d_out, bias=False)          # neighbour transform (shared with self term for GCN)
        if kind == 'gcn':
            self.bias = nn.Parameter(torch.zeros(d_out))
            nn.init.xavier_uniform_(self.lin.weight)
        else:
            self.root = nn.Linear(d_in, d_out, bias=True)      # separate root transform (GraphConv / SAGE)

    def forward(self, G, h):                                    # ordinary layer
        if self.kind == 'gcn':
            hw = self.lin(h)
            return G.agg(hw) + G.self_w.unsqueeze(1) * hw + self.bias
        return G.agg(self.lin(h)) + self.root(h)

    def neighbours(self, G, h_stage1):                          # aggregated messages from stage-one reps
        return G.agg(self.lin(h_stage1))

    def centre_pair(self, h_mix, sw_i, sw_j):                   # self terms for branch i and branch j
        if self.kind == 'gcn':
            hw = self.lin(h_mix)
            return sw_i.unsqueeze(1) * hw + self.bias, sw_j.unsqueeze(1) * hw + self.bias
        c = self.root(h_mix)
        return c, c


class TwoStageMixupGNN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, num_layers, kind='gcn', jk=None, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList([MixConv(in_dim if l == 0 else hidden, hidden, kind) for l in range(num_layers)])
        self.kind, self.jk, self.dropout = kind, jk, dropout
        self.cls = nn.Linear(hidden * num_layers if jk == 'cat' else hidden, out_dim)

    def _post(self, h):
        return F.dropout(F.relu(h), self.dropout, self.training)

    def _readout(self, hs):
        if self.jk == 'cat':
            h = torch.cat(hs, dim=1)
        elif self.jk == 'max':
            h = torch.stack(hs, 0).max(0).values
        else:
            h = hs[-1]
        return self.cls(h)

    def forward(self, x, G):
        hs, h = [], x
        for conv in self.convs:
            h = self._post(conv(G, h))
            hs.append(h)
        return self._readout(hs)

    def forward_mixup(self, x, G, lam, perm):
        # ---- Stage 1: feed-forward without Mixup (Alg. 1, lines 1-8) ----
        stage1, h = [x], x
        for conv in self.convs[:-1]:
            h = self._post(conv(G, h))
            stage1.append(h)
        # ---- Stage 2: two-branch Mixup graph convolutions (lines 9-20) ----
        h_mix = lam * x + (1 - lam) * x[perm]                   # Eq. (5)
        sw_i, sw_j = G.self_w, G.self_w[perm]
        outs = []
        for l, conv in enumerate(self.convs):
            nb = conv.neighbours(G, stage1[l])                   # messages from stage-one neighbours
            c_i, c_j = conv.centre_pair(h_mix, sw_i, sw_j)
            h_i = F.relu(nb + c_i)                               # branch on node i's topology   (Eq. 6)
            h_j = F.relu(nb[perm] + c_j)                         # branch on node j's topology   (Eq. 6)
            h_mix = F.dropout(lam * h_i + (1 - lam) * h_j, self.dropout, self.training)   # Eq. (7)
            outs.append(h_mix)
        return self._readout(outs)


class GATNet(nn.Module):
    """2-layer GAT baseline (Velickovic et al., 2018) for Table 3."""
    def __init__(self, in_dim, out_dim, hidden=8, heads=8, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        self.c1 = GATConv(in_dim, hidden, heads=heads, dropout=dropout)
        self.c2 = GATConv(hidden * heads, out_dim, heads=1, concat=False, dropout=dropout)

    def forward(self, x, edge_index):
        x = F.dropout(x, self.dropout, self.training)
        x = F.elu(self.c1(x, edge_index))
        x = F.dropout(x, self.dropout, self.training)
        return self.c2(x, edge_index)

print('models ready')

---
## 5. Sanity check against the authors' released code
The class below re-creates `Net` and `GraphConv` from the authors' repository (logic unchanged; ported to current PyG). We copy its weights into our model and compare outputs for (a) the plain forward and (b) the two-stage Mixup forward with the same λ and the same node pairing. Dropout is disabled so both are deterministic. The maximum absolute difference should be at floating-point level (≈1e-6). This shows that our GraphConv backbone reproduces the released method exactly, and our GCN/JKNet variants only swap the layer type.

In [ ]:
from torch_geometric.nn.conv import MessagePassing


class AuthorGraphConv(MessagePassing):                  # graph_conv.py from the authors' repo
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='mean')
        self.weight = nn.Parameter(torch.empty(in_channels, out_channels))
        self.lin = nn.Linear(in_channels, out_channels)
        bound = 1.0 / math.sqrt(in_channels)
        nn.init.uniform_(self.weight, -bound, bound)

    def forward(self, x, edge_index, x_cen):
        h = torch.matmul(x, self.weight)
        return self.propagate(edge_index, h=h) + self.lin(x_cen)

    def message(self, h_j):
        return h_j


class AuthorNet(nn.Module):                               # Net from mixup.py (3 GraphConv layers + linear)
    def __init__(self, hidden, in_c, out_c):
        super().__init__()
        self.conv1, self.conv2, self.conv3 = AuthorGraphConv(in_c, hidden), AuthorGraphConv(hidden, hidden), AuthorGraphConv(hidden, hidden)
        self.lin = nn.Linear(hidden, out_c)

    def forward(self, x0, edge_index, edge_index_b, lam, id_new_value_old):
        x1 = F.dropout(F.relu(self.conv1(x0, edge_index, x0)), 0.4, self.training)
        x2 = F.dropout(F.relu(self.conv2(x1, edge_index, x1)), 0.4, self.training)
        x0_b, x1_b, x2_b = x0[id_new_value_old], x1[id_new_value_old], x2[id_new_value_old]
        x0_mix = x0 * lam + x0_b * (1 - lam)
        new_x1, new_x1_b = F.relu(self.conv1(x0, edge_index, x0_mix)), F.relu(self.conv1(x0_b, edge_index_b, x0_mix))
        x1_mix = F.dropout(new_x1 * lam + new_x1_b * (1 - lam), 0.4, self.training)
        new_x2, new_x2_b = F.relu(self.conv2(x1, edge_index, x1_mix)), F.relu(self.conv2(x1_b, edge_index_b, x1_mix))
        x2_mix = F.dropout(new_x2 * lam + new_x2_b * (1 - lam), 0.4, self.training)
        new_x3, new_x3_b = F.relu(self.conv3(x2, edge_index, x2_mix)), F.relu(self.conv3(x2_b, edge_index_b, x2_mix))
        x3_mix = F.dropout(new_x3 * lam + new_x3_b * (1 - lam), 0.4, self.training)
        return self.lin(x3_mix).log_softmax(dim=-1)


set_seed(0)
_ds = Planetoid(osp.join(DATA_DIR, 'Planetoid'), 'Cora', transform=T.NormalizeFeatures())[0]
_n = _ds.num_nodes
_author = AuthorNet(64, _ds.num_features, 7).eval()
_ours = TwoStageMixupGNN(_ds.num_features, 64, 7, num_layers=3, kind='graphconv', dropout=0.4).eval()
with torch.no_grad():                                   # copy the authors' weights into our model
    for mine, theirs in zip(_ours.convs, [_author.conv1, _author.conv2, _author.conv3]):
        mine.lin.weight.copy_(theirs.weight.t())
        mine.root.load_state_dict(theirs.lin.state_dict())
    _ours.cls.load_state_dict(_author.lin.state_dict())

# node pairing exactly as in the authors' shuffleData / idNode
_train = torch.from_numpy(np.random.permutation(_n)[: int(0.6 * _n)])
_perm = torch.arange(_n)
_perm[_train] = _train[torch.randperm(_train.numel())]
_inv = torch.empty_like(_perm)
_inv[_perm] = torch.arange(_n)
_edge_b = _inv[_ds.edge_index]                          # relabelled edge_index_b of the authors' code
_rowptr, _col = edge_index_to_csr(_ds.edge_index, _n)
_G = make_graph(_rowptr, _col, _n, 'graphconv', 'cpu')

with torch.no_grad():
    for lam in [1.0, 0.37]:
        a = _author(_ds.x, _ds.edge_index, _edge_b if lam < 1 else _ds.edge_index, lam, _perm if lam < 1 else torch.arange(_n))
        b = (_ours.forward_mixup(_ds.x, _G, lam, _perm) if lam < 1 else _ours(_ds.x, _G)).log_softmax(-1)
        diff = (a - b).abs().max().item()
        print(f'lambda={lam:<5}  max |authors - ours| = {diff:.2e}   ->  {"MATCH" if diff < 1e-4 else "MISMATCH"}')
        assert diff < 1e-4, 'implementation does not match the authors code'

---
## 6. Table 3 – transductive node classification (Citeseer, Cora, Pubmed)
### 6.1 Data and splits
Planetoid citation graphs with row-normalised features (as in the authors' code). Following the paper, nodes are split randomly into 60 % train / 20 % validation / 20 % test; split *s* uses a fixed seed so every method sees the same 10 splits. The splits are saved to `OUT_DIR/splits/` for the record.

In [ ]:
from torch_geometric.datasets import Amazon, Coauthor

PYG_NAME = {'Cora': 'Cora', 'Citeseer': 'CiteSeer', 'Pubmed': 'PubMed'}
_T3_CACHE = {}


def load_planetoid(name):
    """Full-graph node-classification datasets: Planetoid (Table 3) and the Section 8 new datasets."""
    if name in _T3_CACHE:
        return _T3_CACHE[name]
    if name in PYG_NAME:
        ds = Planetoid(osp.join(DATA_DIR, 'Planetoid'), PYG_NAME[name], transform=T.NormalizeFeatures())
    elif name in ('Computers', 'Photo'):
        ds = Amazon(osp.join(DATA_DIR, 'Amazon'), name, transform=T.NormalizeFeatures())
    elif name in ('CS', 'Physics'):
        ds = Coauthor(osp.join(DATA_DIR, 'Coauthor'), name, transform=T.NormalizeFeatures())
    else:
        raise ValueError(f'unknown dataset {name}')
    d = ds[0]
    rowptr, col = edge_index_to_csr(d.edge_index, d.num_nodes)
    D = SimpleNamespace(name=name, x=d.x, y=d.y, edge_index=d.edge_index, rowptr=rowptr, col=col,
                        n=d.num_nodes, num_features=ds.num_features, num_classes=ds.num_classes, graphs={})
    _T3_CACHE[name] = D
    return D


def make_split(n, split_id, train=0.6, val=0.2):
    rng = np.random.RandomState(1000 + split_id)
    perm = rng.permutation(n)
    a, b = int(train * n), int((train + val) * n)
    return {k: torch.from_numpy(v).long() for k, v in
            {'train': perm[:a], 'val': perm[a:b], 'test': perm[b:]}.items()}


os.makedirs(osp.join(OUT_DIR, 'splits'), exist_ok=True)
rows = []
for name in T3_DATASETS:
    D = load_planetoid(name)
    for s in range(P['t3_splits']):
        sp = make_split(D.n, s)
        np.savez(osp.join(OUT_DIR, 'splits', f'{name}_split{s}.npz'), **{k: v.numpy() for k, v in sp.items()})
    rows.append(dict(dataset=name, nodes=D.n, directed_edges=D.col.numel(), classes=D.num_classes,
                     features=D.num_features, train=len(sp['train']), val=len(sp['val']), test=len(sp['test'])))
pd.DataFrame(rows)

### 6.2 Methods and the training loop
* **GCN / JKNet:** the backbone above with `kind='gcn'` (JKNet adds `jk='cat'`).
* **DropEdge+X:** a fresh random 30 % of undirected edges is removed at every training step and the adjacency is re-normalised.
* **Mixup+X:** a new λ and a new random pairing of the training nodes at every step (as in the authors' code).

Each run trains for a fixed number of epochs, evaluates every epoch, and keeps the test accuracy at the epoch with the best validation accuracy (ties broken by lower validation loss). The last-epoch test accuracy is also stored because the authors' script prints that value.

In [ ]:
T3_SPECS = {
    'GCN':            dict(arch='mix', kind=T3_CONV, jk=None,  mixup=False, dropedge=False),
    'GAT':            dict(arch='gat', fixed_depth=2),
    'JKNet':          dict(arch='mix', kind='gcn',   jk='cat', mixup=False, dropedge=False),
    'DropEdge+GCN':   dict(arch='mix', kind=T3_CONV, jk=None,  mixup=False, dropedge=True),
    'DropEdge+JKNet': dict(arch='mix', kind='gcn',   jk='cat', mixup=False, dropedge=True),
    'Mixup+GCN':      dict(arch='mix', kind=T3_CONV, jk=None,  mixup=True,  dropedge=False),
    'Mixup+JKNet':    dict(arch='mix', kind='gcn',   jk='cat', mixup=True,  dropedge=False),
    # authors' repository configuration (Section 7): GraphConv layer, 3 layers, Beta(4, 4)
    'Repo:GraphConv':       dict(arch='mix', kind='graphconv', jk=None, mixup=False, dropedge=False, fixed_depth=3),
    'Repo:Mixup+GraphConv': dict(arch='mix', kind='graphconv', jk=None, mixup=True,  dropedge=False, fixed_depth=3, alpha=4.0),
}


def full_graph(D, kind):
    if kind not in D.graphs:
        D.graphs[kind] = make_graph(D.rowptr, D.col, D.n, kind, DEVICE)
    return D.graphs[kind]


def train_transductive(D, split, method, depth, seed, hp=None):
    spec = T3_SPECS[method]
    set_seed(seed)
    x, y = D.x.to(DEVICE), D.y.to(DEVICE)
    tr, va, te = (split[k].to(DEVICE) for k in ('train', 'val', 'test'))

    if spec['arch'] == 'gat':
        hp = hp or GAT_HP
        model = GATNet(D.num_features, D.num_classes, hp['hidden'], hp['heads'], hp['dropout']).to(DEVICE)
        ei = D.edge_index.to(DEVICE)
        fwd_eval = lambda: model(x, ei)
    else:
        hp = hp or T3_HP
        model = TwoStageMixupGNN(D.num_features, hp['hidden'], D.num_classes, depth,
                                 spec['kind'], spec['jk'], hp['dropout']).to(DEVICE)
        G_full = full_graph(D, spec['kind'])
        fwd_eval = lambda: model(x, G_full)
    opt = torch.optim.Adam(model.parameters(), lr=hp['lr'], weight_decay=hp['weight_decay'])
    alpha = spec.get('alpha', MIXUP_ALPHA)

    best = dict(val=-1.0, val_loss=float('inf'), test=0.0, epoch=0)
    t0 = time.time()
    for epoch in range(1, hp['epochs'] + 1):
        model.train()
        opt.zero_grad()
        if spec['arch'] == 'gat':
            out = model(x, ei)
            loss = F.cross_entropy(out[tr], y[tr])
        else:
            G = G_full
            if spec['dropedge']:
                rp, cl = edge_index_to_csr(drop_edge_undirected(D.edge_index, DROPEDGE_P), D.n)
                G = make_graph(rp, cl, D.n, spec['kind'], DEVICE)
            if spec['mixup']:
                lam = float(np.random.beta(alpha, alpha))
                perm = torch.arange(D.n, device=DEVICE)
                perm[tr] = tr[torch.randperm(tr.numel(), device=DEVICE)]
                out = model.forward_mixup(x, G, lam, perm)
                loss = lam * F.cross_entropy(out[tr], y[tr]) + (1 - lam) * F.cross_entropy(out[tr], y[perm[tr]])
            else:
                out = model(x, G)
                loss = F.cross_entropy(out[tr], y[tr])
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            out = fwd_eval()
            pred = out.argmax(1)
            val_acc = (pred[va] == y[va]).float().mean().item()
            test_acc = (pred[te] == y[te]).float().mean().item()
            val_loss = F.cross_entropy(out[va], y[va]).item()
        if val_acc > best['val'] or (val_acc == best['val'] and val_loss < best['val_loss']):
            best = dict(val=val_acc, val_loss=val_loss, test=test_acc, epoch=epoch)
    return dict(val=100 * best['val'], test=100 * best['test'], test_last=100 * test_acc,
                best_epoch=best['epoch'], secs=round(time.time() - t0, 2))

print('Table 3 training loop ready')

### 6.3 Experiment runner
Runs every (dataset, method, depth, split, seed) combination that is not yet in the result store. Seed = `100 * split + trial`.

In [ ]:
def run_transductive_grid(tag, datasets, methods, depths, n_splits, n_seeds, hp=None):
    jobs = []
    for name in datasets:
        for m in methods:
            spec = T3_SPECS[m]
            for L in ([spec['fixed_depth']] if 'fixed_depth' in spec else depths):
                for s in range(n_splits):
                    for t in range(n_seeds):
                        key = f'{tag}|{name}|{m}|L{L}|s{s}|t{t}'
                        if key not in STORE:
                            jobs.append((key, name, m, L, s, t))
    print(f'{tag}: {len(jobs)} runs to do')
    for key, name, m, L, s, t in tqdm(jobs, desc=tag):
        D = load_planetoid(name)
        rec = train_transductive(D, make_split(D.n, s), m, L, seed=100 * s + t, hp=hp)
        STORE.put(key, dict(rec, table=tag, dataset=name, method=m, depth=L, split=s, trial=t))


def summarise(tag, metric='test'):
    """Mean ± std over runs; the depth is chosen per (dataset, method) by mean validation score."""
    df = STORE.frame(tag + '|')
    if df.empty:
        return df, df
    g = (df.groupby(['dataset', 'method', 'depth'])
           .agg(val=('val', 'mean'), mean=(metric, 'mean'), std=(metric, 'std'), runs=(metric, 'size'))
           .reset_index())
    g['std'] = g['std'].fillna(0.0)
    best = g.loc[g.groupby(['dataset', 'method'])['val'].idxmax()].reset_index(drop=True)
    return best, g

### 6.4 One-minute demo (for the update video)
A short, self-contained run to show on screen: GCN vs. Mixup+GCN on **Cora**, 2 layers, 3 of the random splits, 300 epochs each. On a T4 this takes well under a minute. It prints the validation-selected test accuracy of each run and the average Mixup gain, next to the paper's values. This demo is **not** stored in the results file; the real numbers come from the full runs below.

In [ ]:
DEMO_DATASET, DEMO_SPLITS, DEMO_DEPTH = 'Cora', 3, 2
_D = load_planetoid(DEMO_DATASET)
print(f'{DEMO_DATASET}: {_D.n} nodes, {_D.col.numel() // 2} undirected edges, {_D.num_classes} classes, '
      f'{_D.num_features} features | device: {DEVICE}\n')
_demo, _t0 = [], time.time()
for s in range(DEMO_SPLITS):
    for m in ['GCN', 'Mixup+GCN']:
        r = train_transductive(_D, make_split(_D.n, s), m, DEMO_DEPTH, seed=100 * s)
        _demo.append(dict(split=s, method=m, val_acc=round(r['val'], 2), test_acc=round(r['test'], 2),
                          best_epoch=r['best_epoch'], seconds=r['secs']))
        print(f"split {s}  {m:<10}  val {r['val']:.2f}  test {r['test']:.2f}  (best epoch {r['best_epoch']}, {r['secs']:.1f}s)")
_demo = pd.DataFrame(_demo)
_mean = _demo.groupby('method')['test_acc'].mean()
print(f"\nMean test accuracy   GCN {_mean['GCN']:.2f}  |  Mixup+GCN {_mean['Mixup+GCN']:.2f}  "
      f"|  Mixup gain {_mean['Mixup+GCN'] - _mean['GCN']:+.2f}")
print('Paper (Table 3, Cora) GCN 88.3  |  Mixup+GCN 90.0  |  Mixup gain +1.7')
print(f'Demo finished in {time.time() - _t0:.0f}s')

### 6.5 Run Table 3
Medium mode = 10 splits × 3 seeds × 4 depths per method (GAT uses a fixed 2 layers). Progress is saved after every run, so this cell can be interrupted and re-run.

In [ ]:
if RUN_TABLE3:
    run_transductive_grid('T3', T3_DATASETS, T3_METHODS, P['t3_depths'], P['t3_splits'], P['t3_seeds'])

### 6.6 Table 3 – reproduced vs. paper
Each cell shows **ours** (mean ± std over runs, best depth by validation) next to the paper's value. `Δ` = ours − paper. The second table compares the **Mixup gain** over the corresponding backbone, which is the paper's central claim.

In [ ]:
PAPER_T3 = {   # Table 3 of the paper: test accuracy (%), mean ± std
    'GCN':            {'Citeseer': (77.1, 1.4), 'Cora': (88.3, 0.8), 'Pubmed': (86.4, 1.1)},
    'GAT':            {'Citeseer': (76.3, 0.8), 'Cora': (87.6, 0.5), 'Pubmed': (85.7, 0.7)},
    'JKNet':          {'Citeseer': (78.1, 0.9), 'Cora': (89.1, 1.2), 'Pubmed': (86.9, 1.3)},
    'LGCN':           {'Citeseer': (77.5, 1.1), 'Cora': (89.0, 1.2), 'Pubmed': (86.5, 0.6)},
    'GMNN':           {'Citeseer': (77.4, 1.5), 'Cora': (88.7, 0.8), 'Pubmed': (86.7, 1.0)},
    'ResGCN':         {'Citeseer': (77.9, 0.8), 'Cora': (88.1, 0.6), 'Pubmed': (87.1, 1.2)},
    'DropEdge+GCN':   {'Citeseer': (78.1, 1.1), 'Cora': (89.2, 0.7), 'Pubmed': (87.3, 0.6)},
    'DropEdge+JKNet': {'Citeseer': (79.3, 0.7), 'Cora': (89.9, 0.8), 'Pubmed': (87.6, 0.9)},
    'Mixup+GCN':      {'Citeseer': (78.7, 0.9), 'Cora': (90.0, 0.7), 'Pubmed': (87.9, 0.8)},
    'Mixup+JKNet':    {'Citeseer': (80.1, 0.8), 'Cora': (90.4, 0.9), 'Pubmed': (88.3, 0.6)},
}
PAPER_T4 = {   # Table 4 of the paper: test F1-micro (%), mean ± std
    'GraphSAGE-mean':      {'Flickr': (50.1, 1.1), 'Yelp': (63.4, 0.6), 'Amazon': (75.8, 0.2)},
    'GraphSAGE-LSTM':      {'Flickr': (50.3, 1.3), 'Yelp': (63.2, 0.8), 'Amazon': (75.7, 0.1)},
    'GraphSAGE-pool':      {'Flickr': (50.0, 0.8), 'Yelp': (63.1, 0.5), 'Amazon': (75.5, 0.2)},
    'DropEdge+GraphSAGE':  {'Flickr': (50.8, 0.9), 'Yelp': (64.1, 0.8), 'Amazon': (76.4, 0.1)},
    'Mixup+GraphSAGE':     {'Flickr': (51.6, 0.8), 'Yelp': (64.6, 0.6), 'Amazon': (77.3, 0.1)},
    'GraphSAINT-GCN':      {'Flickr': (51.1, 0.2), 'Yelp': (65.3, 0.3), 'Amazon': (81.5, 0.1)},
    'GraphSAINT-GAT':      {'Flickr': (50.5, 0.1), 'Yelp': (65.1, 0.2), 'Amazon': (81.5, 0.1)},
    'GraphSAINT-JKNet':    {'Flickr': (51.3, 0.5), 'Yelp': (65.3, 0.4), 'Amazon': (81.6, 0.1)},
    'DropEdge+GraphSAINT': {'Flickr': (51.7, 0.6), 'Yelp': (65.8, 0.7), 'Amazon': (81.8, 0.2)},
    'Mixup+GraphSAINT':    {'Flickr': (52.4, 0.4), 'Yelp': (66.3, 0.4), 'Amazon': (82.0, 0.1)},
}


def comparison_table(best, paper, datasets, method_order):
    rows = []
    for m in method_order:
        row = {'Method': m}
        for d in datasets:
            p = paper.get(m, {}).get(d)
            r = best[(best.dataset == d) & (best.method == m)] if not best.empty else best
            row[f'{d} paper'] = f'{p[0]:.1f}±{p[1]:.1f}' if p else '—'
            if len(r):
                r = r.iloc[0]
                row[f'{d} ours'] = f"{r['mean']:.1f}±{r['std']:.1f} (L={int(r['depth'])}, n={int(r['runs'])})"
                row[f'{d} Δ'] = f"{r['mean'] - p[0]:+.1f}" if p else '—'
            else:
                row[f'{d} ours'], row[f'{d} Δ'] = 'not run', '—'
        rows.append(row)
    return pd.DataFrame(rows).set_index('Method')


def mixup_gain_table(best, paper, datasets, pairs):
    if best.empty:
        return pd.DataFrame()
    rows = []
    for base, mix in pairs:
        for d in datasets:
            get = lambda m: best[(best.dataset == d) & (best.method == m)]['mean']
            ours = (get(mix).iloc[0] - get(base).iloc[0]) if len(get(mix)) and len(get(base)) else np.nan
            pp = paper[mix][d][0] - paper[base][d][0] if d in paper.get(mix, {}) else np.nan
            rows.append({'Comparison': f'{mix} − {base}', 'Dataset': d,
                         'Paper gain': round(pp, 2), 'Our gain': round(ours, 2),
                         'Same direction': ('yes' if np.sign(ours) == np.sign(pp) else 'no') if not np.isnan(ours) else '—'})
    return pd.DataFrame(rows)


T3_ORDER = ['GCN', 'GAT', 'JKNet', 'LGCN', 'GMNN', 'ResGCN', 'DropEdge+GCN', 'DropEdge+JKNet', 'Mixup+GCN', 'Mixup+JKNet']
best_t3, all_t3 = summarise('T3')
table3 = comparison_table(best_t3, PAPER_T3, T3_DATASETS, T3_ORDER)
display(table3)
gain_t3 = mixup_gain_table(best_t3, PAPER_T3, T3_DATASETS,
                           [('GCN', 'Mixup+GCN'), ('JKNet', 'Mixup+JKNet'), ('DropEdge+GCN', 'Mixup+GCN')])
display(gain_t3)

### 6.7 Effect of depth (Table 3)
The paper searches depth on the validation set. This view shows validation and test accuracy for every depth that was run, so the choice is transparent.

In [ ]:
if not all_t3.empty:
    depth_view = all_t3.pivot_table(index=['dataset', 'method'], columns='depth', values='mean').round(1)
    display(depth_view)

---
## 7. Authors' repository configuration (continuity with the Pubmed pilot)
The released `mixup.py` differs from the paper's description in a few ways: it uses the **GraphConv** layer (mean aggregation + separate root weight) rather than GCN, exactly **3 layers**, **Beta(4, 4)** instead of α = 1, and it prints the **last-epoch** test accuracy with a single seed. This section runs that configuration with several seeds so the pilot numbers from the proposal (Pubmed: 87.85 → 89.10) can be compared with a mean ± std, reported both the repo's way (last epoch) and with validation-based selection.

In [ ]:
if RUN_REPO_CONFIG:
    run_transductive_grid('REPO', REPO_DATASETS, ['Repo:GraphConv', 'Repo:Mixup+GraphConv'],
                          depths=[3], n_splits=1, n_seeds=REPO_SEEDS)
    df = STORE.frame('REPO|')
    if not df.empty:
        repo_tab = (df.groupby(['dataset', 'method'])
                      .agg(last_epoch_mean=('test_last', 'mean'), last_epoch_std=('test_last', 'std'),
                           val_selected_mean=('test', 'mean'), val_selected_std=('test', 'std'), runs=('test', 'size'))
                      .round(2))
        display(repo_tab)
        print('Pilot run in the proposal (Pubmed, single run): GraphConv 87.85, Mixup+GraphConv 89.10')

---
## 8. New data: existing datasets not used in the paper (Amazon Computers, Amazon Photo)
This is the first **new-data** part of the project (proposal, Section 5 / Table 4 there). Both graphs come from the `gnn-benchmark` collection of Shchur et al. (2018), *Pitfalls of Graph Neural Network Evaluation*, and are built from Amazon co-purchase data (McAuley et al., 2015):
* **Nodes** are products, **edges** connect products that are frequently bought together, **features** are bag-of-words vectors of product reviews, and the **label** is the product category.
* They are a different domain from the citation graphs of Table 3 (e-commerce instead of science), and denser, which makes them a useful test of whether the Mixup gain transfers.

**Protocol.** Exactly the Table 3 protocol, so the results are directly comparable: row-normalised features, random 60/20/20 splits (`NEW_SPLITS` splits × `NEW_SEEDS` seeds), depth chosen on validation from `NEW_DEPTHS`, test accuracy at the best validation epoch, α = 1. The one change is **500 epochs** instead of 300 (`NEW_EPOCHS`), because a pilot run on Photo was still improving at epoch 300. Only the backbone-vs-Mixup pairs are run (GCN, JKNet), because the question here is whether Mixup still improves its backbone. There are no paper numbers for these datasets.

In [ ]:
if RUN_NEW_DATA:
    rows = []
    for name in NEW_DATASETS:
        D = load_planetoid(name)
        deg = D.rowptr.diff().float()
        rows.append(dict(dataset=name, nodes=D.n, undirected_edges=D.col.numel() // 2, classes=D.num_classes,
                         features=D.num_features, avg_degree=round(deg.mean().item(), 1),
                         isolated_nodes=int((deg == 0).sum()),
                         largest_class_share=f'{100 * torch.bincount(D.y).max().item() / D.n:.1f}%'))
    new_stats = pd.DataFrame(rows).set_index('dataset')
    new_stats.to_csv(osp.join(OUT_DIR, 'new_data_statistics.csv'))
    display(new_stats)

### 8.1 Run the new-data experiments

In [ ]:
if RUN_NEW_DATA:
    NEW_HP = dict(T3_HP, epochs=NEW_EPOCHS)
    run_transductive_grid('NEW', NEW_DATASETS, NEW_METHODS, NEW_DEPTHS, NEW_SPLITS, NEW_SEEDS, hp=NEW_HP)

### 8.2 New-data results and Mixup gain
Same format as Table 3 (mean ± std over runs, best depth by validation). The gain table shows the difference between each Mixup model and its backbone, with the paper's average gain on the citation datasets for reference.

In [ ]:
def results_table(best, datasets, methods):
    rows = []
    for m in methods:
        row = {'Method': m}
        for d in datasets:
            r = best[(best.dataset == d) & (best.method == m)] if not best.empty else best
            row[d] = (f"{r.iloc[0]['mean']:.1f}±{r.iloc[0]['std']:.1f} (L={int(r.iloc[0]['depth'])}, n={int(r.iloc[0]['runs'])})"
                      if len(r) else 'not run')
        rows.append(row)
    return pd.DataFrame(rows).set_index('Method')


def gain_only_table(best, datasets, pairs, reference):
    rows = []
    for base, mix in pairs:
        for d in datasets:
            get = lambda m: best[(best.dataset == d) & (best.method == m)] if not best.empty else best
            b, x = get(base), get(mix)
            if len(b) and len(x):
                g = x.iloc[0]['mean'] - b.iloc[0]['mean']
                pooled = math.sqrt((x.iloc[0]['std'] ** 2 + b.iloc[0]['std'] ** 2) / 2)
                rows.append({'Comparison': f'{mix} − {base}', 'Dataset': d, 'Our gain': round(g, 2),
                             'Gain / pooled std': round(g / pooled, 2) if pooled > 0 else np.nan,
                             'Paper avg gain on citation data': reference[(base, mix)]})
    return pd.DataFrame(rows)


best_new, all_new = summarise('NEW')
table_new = results_table(best_new, NEW_DATASETS, NEW_METHODS)
display(table_new)
_ref = {(b, m): round(np.mean([PAPER_T3[m][d][0] - PAPER_T3[b][d][0] for d in T3_DATASETS]), 2)
        for b, m in [('GCN', 'Mixup+GCN'), ('JKNet', 'Mixup+JKNet')]}
gain_new = gain_only_table(best_new, NEW_DATASETS, [('GCN', 'Mixup+GCN'), ('JKNet', 'Mixup+JKNet')], _ref)
display(gain_new)
if not all_new.empty:
    display(all_new.pivot_table(index=['dataset', 'method'], columns='depth', values='mean').round(1))

---
## 9. Table 4 – inductive node classification (Flickr, Yelp, Amazon)
### 9.1 Data
The three graphs come from the GraphSAINT release (Zeng et al., 2020) with its fixed train/val/test partition. The files are downloaded from the same Google Drive IDs that PyG uses, but they are loaded directly as CSR arrays, which avoids PyG's very memory-hungry processing step for Amazon. Features are standardised with training-node statistics, as in GraphSAINT.

**Inductive setting:** training only sees the sub-graph induced by the training nodes. Validation and test predictions use the full graph.

| Dataset | Nodes | Edges | Classes | Labels | Download size |
|---|---|---|---|---|---|
| Flickr | 89,250 | 899,756 | 7 | single | ~0.5 GB |
| Yelp | 716,847 | 6,977,410 | 100 | multi | ~2 GB |
| Amazon | 1,598,960 | 132,169,734 | 107 | multi | ~5 GB |

In [ ]:
from torch_geometric.data import download_google_url
from torch_geometric.datasets import Flickr, Yelp, AmazonProducts

SAINT_SOURCES = {'Flickr': Flickr, 'Yelp': Yelp, 'Amazon': AmazonProducts}
_T4_CACHE = {}


def load_saint(name):
    if name in _T4_CACHE:
        return _T4_CACHE[name]
    root = osp.join(DATA_DIR, 'graphsaint', name)
    raw, proc = osp.join(root, 'raw'), osp.join(root, 'prepared.pt')
    if not osp.exists(proc):
        src = SAINT_SOURCES[name]
        for fname, attr in [('adj_full.npz', 'adj_full_id'), ('feats.npy', 'feats_id'),
                            ('class_map.json', 'class_map_id'), ('role.json', 'role_id')]:
            if not osp.exists(osp.join(raw, fname)):
                download_google_url(getattr(src, attr), raw, fname)
        print(f'[{name}] building arrays ...')
        f = np.load(osp.join(raw, 'adj_full.npz'))
        rowptr = torch.from_numpy(f['indptr'].astype(np.int64))
        col = torch.from_numpy(f['indices'].astype(np.int64))
        del f
        x = np.load(osp.join(raw, 'feats.npy')).astype(np.float32)
        n = x.shape[0]
        with open(osp.join(raw, 'class_map.json')) as fh:
            cmap = json.load(fh)
        first = next(iter(cmap.values()))
        multilabel = isinstance(first, list)
        if multilabel:
            y = np.zeros((n, len(first)), dtype=np.uint8)
            for k, v in cmap.items():
                y[int(k)] = v
        else:
            y = np.zeros(n, dtype=np.int64)
            for k, v in cmap.items():
                y[int(k)] = v
        del cmap
        with open(osp.join(raw, 'role.json')) as fh:
            role = json.load(fh)
        tr, va, te = (np.asarray(role[k], dtype=np.int64) for k in ('tr', 'va', 'te'))
        mu, sd = x[tr].mean(0), x[tr].std(0)
        sd[sd == 0] = 1.0
        x = (x - mu) / sd
        torch.save(dict(rowptr=rowptr, col=col, x=torch.from_numpy(x), y=torch.from_numpy(y),
                        train_idx=torch.from_numpy(tr), val_idx=torch.from_numpy(va), test_idx=torch.from_numpy(te),
                        multilabel=multilabel), proc)
        del x, y, rowptr, col
    d = torch.load(proc)
    D = SimpleNamespace(name=name, **d)
    D.n = D.x.size(0)
    D.num_features = D.x.size(1)
    D.num_classes = D.y.size(1) if D.multilabel else int(D.y.max()) + 1
    # ---- inductive training graph: keep only edges whose two endpoints are training nodes ----
    is_tr = torch.zeros(D.n, dtype=torch.bool)
    is_tr[D.train_idx] = True
    row = csr_rows(D.rowptr)
    keep = is_tr[row] & is_tr[D.col]
    D.train_col = D.col[keep]
    D.train_rowptr = rowptr_from_rows(row[keep], D.n)
    del row, keep
    D.train_deg = D.train_rowptr.diff()
    D.graphs, D.saint = {}, {}
    _T4_CACHE[name] = D
    return D


rows = []
for name in T4_DATASETS:
    D = load_saint(name)
    rows.append(dict(dataset=name, nodes=D.n, directed_edges=D.col.numel(), train_graph_edges=D.train_col.numel(),
                     classes=D.num_classes, multilabel=D.multilabel, features=D.num_features,
                     train=len(D.train_idx), val=len(D.val_idx), test=len(D.test_idx)))
pd.DataFrame(rows)

### 9.2 Mini-batch samplers (pure PyTorch)
* **GraphSAINT random-walk sampler** (Zeng et al., 2020): `roots` training nodes start random walks of length `walk`; the mini-batch is the sub-graph induced by all visited nodes. With `USE_SAINT_NORM`, the aggregator normalisation $C_v/C_{u,v}$ and loss normalisation are estimated by pre-sampling, as in GraphSAINT.
* **GraphSAGE neighbour sampler** (Hamilton et al., 2017): a batch of seed training nodes plus `fanout[k]` sampled neighbours per node at hop *k*; the loss is computed on the seeds only.

The two-stage Mixup is applied **inside each mini-batch** (Algorithm 1 takes the mini-batch graph as input): training nodes of the batch are randomly paired.

In [ ]:
def induced_subgraph(rowptr, col, nodes, loc):
    """Sub-graph induced by `nodes` on a CSR graph. Returns local CSR plus global edge ids."""
    k = nodes.numel()
    start = rowptr[nodes]
    deg = rowptr[nodes + 1] - start
    tgt = torch.repeat_interleave(torch.arange(k), deg)
    eid = torch.repeat_interleave(start - (deg.cumsum(0) - deg), deg) + torch.arange(tgt.numel())
    loc[nodes] = torch.arange(k)
    src = loc[col[eid]]
    loc[nodes] = -1
    keep = src >= 0
    tgt, src, eid = tgt[keep], src[keep], eid[keep]
    return SimpleNamespace(nodes=nodes, rowptr=rowptr_from_rows(tgt, k), tgt=tgt, col=src, eid=eid, num_loss=k)


class SaintRWSampler:
    def __init__(self, rowptr, col, train_idx, num_roots, walk_length):
        self.rowptr, self.col, self.train_idx = rowptr, col, train_idx
        self.num_roots, self.walk_length = num_roots, walk_length
        self.loc = torch.full((rowptr.numel() - 1,), -1, dtype=torch.long)

    def sample(self):
        cur = self.train_idx[torch.randint(self.train_idx.numel(), (self.num_roots,))]
        visited = [cur]
        for _ in range(self.walk_length):
            start = self.rowptr[cur]
            deg = self.rowptr[cur + 1] - start
            off = torch.minimum((torch.rand(cur.numel()) * deg).long(), (deg - 1).clamp(min=0))
            pos = torch.where(deg > 0, start + off, torch.zeros_like(start))
            cur = torch.where(deg > 0, self.col[pos], cur)
            visited.append(cur)
        return induced_subgraph(self.rowptr, self.col, torch.unique(torch.cat(visited)), self.loc)


class NeighborSubgraphSampler:
    def __init__(self, rowptr, col, train_idx, batch_size, fanouts):
        self.rowptr, self.col, self.train_idx = rowptr, col, train_idx
        self.batch_size, self.fanouts = batch_size, fanouts
        self.loc = torch.full((rowptr.numel() - 1,), -1, dtype=torch.long)

    def epoch(self, max_steps=None):
        order = self.train_idx[torch.randperm(self.train_idx.numel())]
        for i, s in enumerate(range(0, order.numel(), self.batch_size)):
            if max_steps and i >= max_steps:
                break
            yield self.sample(order[s:s + self.batch_size])

    def sample(self, seeds):
        loc, n = self.loc, seeds.numel()
        loc[seeds] = torch.arange(n)
        nodes, T_, S_, frontier = [seeds], [], [], seeds
        for k in self.fanouts:
            start = self.rowptr[frontier]
            deg = self.rowptr[frontier + 1] - start
            has = deg > 0
            f, st, d = frontier[has], start[has], deg[has]
            if f.numel() == 0:
                break
            off = torch.minimum((torch.rand(f.numel(), k) * d.unsqueeze(1)).long(), (d - 1).unsqueeze(1))
            nbr = self.col[st.unsqueeze(1) + off].reshape(-1)
            tg = f.repeat_interleave(k)
            new = torch.unique(nbr[loc[nbr] < 0])
            loc[new] = torch.arange(n, n + new.numel())
            n += new.numel()
            nodes.append(new)
            T_.append(loc[tg])
            S_.append(loc[nbr])
            frontier = new
        nodes = torch.cat(nodes)
        loc[nodes] = -1
        if not T_:                                               # seeds without any training neighbours
            empty = torch.zeros(0, dtype=torch.long)
            return SimpleNamespace(nodes=nodes, rowptr=torch.zeros(n + 1, dtype=torch.long), tgt=empty,
                                   col=empty, eid=None, num_loss=seeds.numel())
        key = torch.unique(torch.cat(T_) * n + torch.cat(S_))   # de-duplicate, sort by target
        tgt, src = key // n, key % n
        return SimpleNamespace(nodes=nodes, rowptr=rowptr_from_rows(tgt, n), tgt=tgt, col=src, eid=None,
                               num_loss=seeds.numel())


def saint_statistics(D, hp):
    """Pre-sample sub-graphs once per dataset to estimate GraphSAINT normalisation and batch size."""
    key = (hp['saint_roots'], hp['saint_walk'])
    if key in D.saint:
        return D.saint[key]
    sampler = SaintRWSampler(D.train_rowptr, D.train_col, D.train_idx, *key)
    node_count = torch.zeros(D.n)
    edge_count = torch.zeros(D.train_col.numel())
    n_tr, num_samples, total = D.train_idx.numel(), 0, 0
    target = SAINT_COVERAGE * n_tr if USE_SAINT_NORM else 5 * hp['saint_roots'] * (hp['saint_walk'] + 1)
    pbar = tqdm(total=target, desc=f'[{D.name}] GraphSAINT pre-sampling', leave=False)
    while total < target:
        sg = sampler.sample()
        node_count[sg.nodes] += 1
        edge_count[sg.eid] += 1
        total += sg.nodes.numel()
        num_samples += 1
        pbar.update(sg.nodes.numel())
    pbar.close()
    stats = SimpleNamespace(node_count=node_count, edge_count=edge_count, num_samples=num_samples,
                            avg_nodes=total / num_samples)
    D.saint[key] = stats
    return stats

print('samplers ready')

### 9.3 Methods and the inductive training loop
* **GraphSAGE-mean:** `graphconv` layer (mean aggregator + root weight) trained with neighbour sampling.
* **GraphSAINT-GCN / GraphSAINT-JKNet:** `gcn` layer (+ JK concatenation) trained on GraphSAINT random-walk sub-graphs with GraphSAINT normalisation.
* **DropEdge+X:** 30 % of the mini-batch edges removed at every step.
* **Mixup+X:** two-stage Mixup inside each mini-batch; for the multi-label datasets the loss is binary cross-entropy against the mixed multi-hot labels.

Validation/test F1-micro is computed on the full graph every `eval_every` epochs; the test score at the best validation F1 is reported.

In [ ]:
T4_SPECS = {
    'GraphSAGE-mean':      dict(sampler='sage',  kind='graphconv', jk=None,  mixup=False, dropedge=False),
    'DropEdge+GraphSAGE':  dict(sampler='sage',  kind='graphconv', jk=None,  mixup=False, dropedge=True),
    'Mixup+GraphSAGE':     dict(sampler='sage',  kind='graphconv', jk=None,  mixup=True,  dropedge=False),
    'GraphSAINT-GCN':      dict(sampler='saint', kind='gcn',       jk=None,  mixup=False, dropedge=False),
    'GraphSAINT-JKNet':    dict(sampler='saint', kind='gcn',       jk='cat', mixup=False, dropedge=False),
    'DropEdge+GraphSAINT': dict(sampler='saint', kind='gcn',       jk=None,  mixup=False, dropedge=True),
    'Mixup+GraphSAINT':    dict(sampler='saint', kind='gcn',       jk=None,  mixup=True,  dropedge=False),
}


def batch_graph(D, sg, spec, stats):
    n, tgt, col, eid = sg.nodes.numel(), sg.tgt, sg.col, sg.eid
    if spec['dropedge']:
        keep = torch.rand(col.numel()) >= DROPEDGE_P
        tgt, col = tgt[keep], col[keep]
        eid = eid[keep] if eid is not None else None
    rowptr = rowptr_from_rows(tgt, n)
    deg, scale = None, None
    if spec['sampler'] == 'saint':
        deg = D.train_deg[sg.nodes].float()                  # full training-graph degree (GraphSAINT convention)
        if spec['dropedge']:
            deg = deg * (1 - DROPEDGE_P)
        if USE_SAINT_NORM:
            cnt = stats.node_count[sg.nodes[tgt]] / stats.edge_count[eid]
            scale = torch.nan_to_num(cnt, nan=0.1, posinf=1e4).clamp(0, 1e4)
    return make_graph(rowptr, col, n, spec['kind'], DEVICE, deg=deg, edge_scale=scale)


def eval_graph(D, kind):
    if kind not in D.graphs:
        D.graphs[kind] = make_graph(D.rowptr, D.col, D.n, kind, DEVICE)   # full graph, all nodes
    return D.graphs[kind]


@torch.no_grad()
def evaluate_inductive(model, D, kind):
    model.eval()
    out = model(D.x_dev, eval_graph(D, kind))
    y = D.y_dev
    return (micro_f1(out[D.val_dev], y[D.val_dev], D.multilabel),
            micro_f1(out[D.test_dev], y[D.test_dev], D.multilabel))


def train_inductive(D, method, depth, seed, hp):
    spec = T4_SPECS[method]
    set_seed(seed)
    if not hasattr(D, 'x_dev'):
        D.x_dev, D.y_dev = D.x.to(DEVICE), D.y.to(DEVICE)
        D.val_dev, D.test_dev = D.val_idx.to(DEVICE), D.test_idx.to(DEVICE)
    model = TwoStageMixupGNN(D.num_features, hp['hidden'], D.num_classes, depth,
                             spec['kind'], spec['jk'], hp['dropout']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=hp['lr'])
    if D.multilabel:
        loss_fn = lambda out, y: F.binary_cross_entropy_with_logits(out, y.float(), reduction='none').mean(1)
    else:
        loss_fn = lambda out, y: F.cross_entropy(out, y, reduction='none')

    stats = None
    if spec['sampler'] == 'saint':
        stats = saint_statistics(D, hp)
        sampler = SaintRWSampler(D.train_rowptr, D.train_col, D.train_idx, hp['saint_roots'], hp['saint_walk'])
        steps = math.ceil(D.train_idx.numel() / stats.avg_nodes)
        if MAX_STEPS_PER_EPOCH:
            steps = min(steps, MAX_STEPS_PER_EPOCH)
        batches = lambda: (sampler.sample() for _ in range(steps))
    else:
        fan = (hp['sage_fanouts'] + [hp['sage_fanouts'][-1]] * depth)[:depth]
        sampler = NeighborSubgraphSampler(D.train_rowptr, D.train_col, D.train_idx, hp['sage_batch'], fan)
        batches = lambda: sampler.epoch(MAX_STEPS_PER_EPOCH)

    best = dict(val=-1.0, test=0.0, epoch=0)
    t0 = time.time()
    for epoch in range(1, hp['epochs'] + 1):
        model.train()
        for sg in batches():
            G = batch_graph(D, sg, spec, stats)
            nodes = sg.nodes.to(DEVICE)
            x, y, nl = D.x_dev[nodes], D.y_dev[nodes], sg.num_loss
            weight = None
            if stats is not None and USE_SAINT_NORM:
                weight = (stats.num_samples / stats.node_count[sg.nodes].clamp(min=0.1) / D.train_idx.numel()).to(DEVICE)
            if spec['mixup']:
                lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
                perm = torch.arange(nodes.numel(), device=DEVICE)
                perm[:nl] = torch.randperm(nl, device=DEVICE)       # pair the loss (training) nodes of the batch
                out = model.forward_mixup(x, G, lam, perm)[:nl]
                l = lam * loss_fn(out, y[:nl]) + (1 - lam) * loss_fn(out, y[perm[:nl]])
            else:
                out = model(x, G)[:nl]
                l = loss_fn(out, y[:nl])
            loss = (l * weight[:nl]).sum() if weight is not None else l.mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        if epoch % hp['eval_every'] == 0 or epoch == hp['epochs']:
            val, test = evaluate_inductive(model, D, spec['kind'])
            if val > best['val']:
                best = dict(val=val, test=test, epoch=epoch)
    return dict(val=100 * best['val'], test=100 * best['test'], test_last=100 * test,
                best_epoch=best['epoch'], secs=round(time.time() - t0, 1))

print('Table 4 training loop ready')

### 9.4 Run Table 4
Datasets are processed one at a time (the previous one is released from memory). Seed = trial index.

In [ ]:
def run_inductive_grid(tag, datasets, methods, depths, n_seeds):
    for name in datasets:
        jobs = [(f'{tag}|{name}|{m}|L{L}|t{t}', m, L, t)
                for m in methods for L in depths for t in range(n_seeds)]
        jobs = [j for j in jobs if j[0] not in STORE]
        print(f'{tag} {name}: {len(jobs)} runs to do')
        if not jobs:
            continue
        D = load_saint(name)
        for key, m, L, t in tqdm(jobs, desc=f'{tag} {name}'):
            rec = train_inductive(D, m, L, seed=t, hp=T4_HP[name])
            STORE.put(key, dict(rec, table=tag, dataset=name, method=m, depth=L, trial=t))
        _T4_CACHE.pop(name, None)                     # free memory before the next dataset
        del D
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


if RUN_TABLE4:
    run_inductive_grid('T4', T4_DATASETS, T4_METHODS, P['t4_depths'], P['t4_seeds'])

### 9.5 Table 4 – reproduced vs. paper

In [ ]:
T4_ORDER = ['GraphSAGE-mean', 'GraphSAGE-LSTM', 'GraphSAGE-pool', 'DropEdge+GraphSAGE', 'Mixup+GraphSAGE',
            'GraphSAINT-GCN', 'GraphSAINT-GAT', 'GraphSAINT-JKNet', 'DropEdge+GraphSAINT', 'Mixup+GraphSAINT']
T4_ALL = ['Flickr', 'Yelp', 'Amazon']
best_t4, all_t4 = summarise('T4')
table4 = comparison_table(best_t4, PAPER_T4, T4_ALL, T4_ORDER)
display(table4)
gain_t4 = mixup_gain_table(best_t4, PAPER_T4, T4_ALL,
                           [('GraphSAGE-mean', 'Mixup+GraphSAGE'), ('GraphSAINT-GCN', 'Mixup+GraphSAINT'),
                            ('DropEdge+GraphSAINT', 'Mixup+GraphSAINT')])
display(gain_t4)

---
## 10. Final comparison, figures and export
### 10.1 Paper vs. reproduction (bar charts)

In [ ]:
def plot_compare(best, paper, datasets, methods, title, fname):
    ds = [d for d in datasets if not best.empty and (best.dataset == d).any()]
    if not ds:
        print('nothing to plot for', title)
        return
    fig, axes = plt.subplots(1, len(ds), figsize=(5.2 * len(ds), 3.8), squeeze=False)
    for ax, d in zip(axes[0], ds):
        ms = [m for m in methods if ((best.dataset == d) & (best.method == m)).any()]
        xs = np.arange(len(ms))
        pv = [paper.get(m, {}).get(d, (np.nan, 0))[0] for m in ms]
        pe = [paper.get(m, {}).get(d, (np.nan, 0))[1] for m in ms]
        ov = [best[(best.dataset == d) & (best.method == m)]['mean'].iloc[0] for m in ms]
        oe = [best[(best.dataset == d) & (best.method == m)]['std'].iloc[0] for m in ms]
        if np.all(np.isnan(pv)):                          # no paper numbers (new datasets): ours only
            ax.bar(xs, ov, 0.55, yerr=oe, label='Ours', color='#2b6cb0', capsize=2)
        else:
            ax.bar(xs - 0.2, pv, 0.4, yerr=pe, label='Paper', color='#9aa5b1', capsize=2)
            ax.bar(xs + 0.2, ov, 0.4, yerr=oe, label='Ours', color='#2b6cb0', capsize=2)
        lo = np.nanmin(pv + ov) - 3
        ax.set_ylim(lo, np.nanmax(pv + ov) + 2)
        ax.set_xticks(xs)
        ax.set_xticklabels(ms, rotation=40, ha='right', fontsize=8)
        ax.set_title(d)
        ax.grid(axis='y', alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False)
    axes[0][0].set_ylabel(title.split(':')[0])
    axes[0][0].legend(frameon=False)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(osp.join(OUT_DIR, fname), dpi=200, bbox_inches='tight')
    plt.show()


plot_compare(best_t3, PAPER_T3, T3_DATASETS, T3_ORDER, 'Test accuracy (%): Table 3, transductive', 'table3_compare.png')
plot_compare(best_t4, PAPER_T4, T4_ALL, T4_ORDER, 'Test F1-micro (%): Table 4, inductive', 'table4_compare.png')
plot_compare(best_new, {}, NEW_DATASETS, NEW_METHODS, 'Test accuracy (%): new datasets (Section 8)', 'new_data_results.png')

### 10.2 Export tables and a reproducibility record
Writes CSV, Markdown and LaTeX versions of both tables, all per-run results, and a JSON record of the environment and every setting, so the numbers in the report can be traced back to this run.

In [ ]:
def export(df, stem):
    if df is None or len(df) == 0:
        return
    df.to_csv(osp.join(OUT_DIR, f'{stem}.csv'))
    for ext, fn in [('md', df.to_markdown), ('tex', df.to_latex)]:
        try:
            with open(osp.join(OUT_DIR, f'{stem}.{ext}'), 'w') as f:
                f.write(fn())
        except ImportError as e:                          # tabulate / jinja2 missing
            print(f'skipped {stem}.{ext}: {e}')


for df, stem in [(table3, 'table3_comparison'), (gain_t3, 'table3_mixup_gain'),
                 (table4, 'table4_comparison'), (gain_t4, 'table4_mixup_gain'),
                 (table_new, 'new_data_results'), (gain_new, 'new_data_mixup_gain')]:
    export(df, stem)
pd.DataFrame(list(STORE.rows.values())).to_csv(osp.join(OUT_DIR, 'all_runs.csv'), index=False)

record = dict(environment=ENV, run_mode=RUN_MODE, t4_override=T4_OVERRIDE, preset=P, mixup_alpha=MIXUP_ALPHA,
              dropedge_p=DROPEDGE_P,
              t3=dict(datasets=T3_DATASETS, methods=T3_METHODS, conv=T3_CONV, hp=T3_HP, gat_hp=GAT_HP),
              new_data=dict(datasets=NEW_DATASETS, methods=NEW_METHODS, splits=NEW_SPLITS, seeds=NEW_SEEDS,
                            depths=NEW_DEPTHS, hp=dict(T3_HP, epochs=NEW_EPOCHS)),
              t4=dict(datasets=T4_DATASETS, methods=T4_METHODS, hp=T4_HP, saint_norm=USE_SAINT_NORM,
                      saint_coverage=SAINT_COVERAGE, max_steps=MAX_STEPS_PER_EPOCH),
              seeds='Table 3: seed = 100*split + trial, split s uses RandomState(1000+s); Table 4: seed = trial',
              finished=time.strftime('%Y-%m-%d %H:%M:%S'))
with open(osp.join(OUT_DIR, 'reproducibility_record.json'), 'w') as f:
    json.dump(record, f, indent=2)
print('Saved to', OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))

### 10.3 Notes for interpreting the results
* **Trials and depth search.** Unless `RUN_MODE='paper'`, fewer trials and depths are run than in the paper. Standard deviations are therefore less stable, and the "best depth" is chosen from a smaller set.
* **Layer type.** The paper reports GCN, but the released code uses a GraphConv layer. `T3_CONV` switches between the two; Section 7 reproduces the released configuration exactly.
* **Mixup α.** The paper states α = 1; the released code uses α = 4. Both are covered (Sections 6 and 7).
* **Model selection.** The released script prints test accuracy every epoch without validation-based selection. This notebook reports the test score at the best validation epoch (and stores the last-epoch score as `test_last`).
* **Table 4 budgets.** Epochs, hidden sizes and batch sizes are scaled to a single Colab GPU. The GraphSAINT samplers and normalisation are re-implemented in plain PyTorch.
* **New datasets (Section 8).** Amazon Computers/Photo use the Table 3 protocol with fewer runs (`NEW_SPLITS` × `NEW_SEEDS`) and depths; there is no paper baseline, so the evidence is the Mixup gain over the same backbone.
* **Not re-implemented.** LGCN, GMNN, ResGCN, GraphSAGE-LSTM/pool and GraphSAINT-GAT appear with the paper's values only.
* **What counts as a successful reproduction.** Exact numbers can differ by about one standard deviation because of hardware, library versions and random splits. The key check is whether Mixup consistently improves its backbone (the *Mixup gain* tables).